# NB42: PAH Panel — EDA Preprocessing + NB21 COMBINED Stratejisi

**Amaç:** `eda.md`'deki EDA bulgularını (hayalet sütun, multicollinearity filtresi) NB21'in
kanıtlanmış COMBINED pooling + BalancedBagging stratejisine ekleyerek fark yaratıp yaratmadığını test etmek.

**Yaklaşım:**
- NB21'in model parametreleri, değerlendirme protokolü, COMBINED havuzu **HİÇ DEĞİŞMEZ**
- Sadece train seti üzerindeki preprocessing'e eda.md adımları eklenir
- Karşılaştırma: NB21 orijinal (Boot_F1=0.582) vs NB42 EDA-temizlikli

**EDA Ek Adımları (eda.md):**
1. Hayalet sütun filtresi (dolu < 30 satır)
2. Multicollinearity filtresi (r > 0.90)

**NB21'den korunan:**
- COMBINED pool (MASTER+KANSER+CFTR → PAH test)
- BalancedBagging(20×LGBM)
- M3 missing (is_missing flag + median imputation)
- Prior-shift + robust %80/20 bootstrap threshold
- LOO-CV değerlendirme

In [1]:
# Cell 1: Imports ve Konfigürasyon
import sys, os, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    matthews_corrcoef, precision_score, recall_score, roc_auc_score
)
from lightgbm import LGBMClassifier
from imblearn.ensemble import BalancedBaggingClassifier

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

from config import SEED, TEST_SIZE, PROJECT_ROOT, REPORTS_DIR
import src.columns_real as CR
from src.metrics import optimize_threshold, compute_all_metrics

np.random.seed(SEED)

# NB21 sabitleri (aynen korunuyor)
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = 123

TARGET_PANEL = "PAH"
RESULTS_DIR_NB = os.path.join(PROJECT_ROOT, "results", "v25_pah_eda_pipeline")
os.makedirs(RESULTS_DIR_NB, exist_ok=True)

print(f'NB42 — PAH EDA + NB21 COMBINED Stratejisi')
print(f'SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}')

NB42 — PAH EDA + NB21 COMBINED Stratejisi
SEED=42, PI_TEST=0.2, N_BOOT=50


In [2]:
# Cell 2: Veri Yükleme (NB21 ile aynı — COMBINED havuz)
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr   = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah    = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# COMBINED eğitim havuzu: MASTER + KANSER + CFTR (PAH HARİÇ — NB21 ile aynı)
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+KANSER+CFTR): {df_combined.shape}")
print(f"  pos={df_combined[TARGET].sum()}, neg={(df_combined[TARGET]==0).sum()}")

# Cross-panel birebir-aynı satır drop (NB21 ile aynı)
feat_cols = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_pah, df_master, feat_cols, TARGET)
if dup_ids:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"PAH: {len(dup_ids)} birebir-aynı satır drop → {df_pah.shape}")
else:
    print("PAH: birebir-aynı satır yok")

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353)   (pos=90, neg=21)
PAH:    (372, 353)  (pos=310, neg=62)

COMBINED (MASTER+KANSER+CFTR): (3430, 353)
  pos=2507, neg=923
PAH: 3 birebir-aynı satır drop → (369, 353)


## Faz 1: Sütun Temizliği (NB21 orijinal + EDA ek adımları)

In [3]:
# Cell 3: Sütun Temizliği — NB21 orijinal (sabit + duplicate) + EDA ek (hayalet + multicollinearity)

# --- NB21 OrijINAL: Sabit sütunlar (MASTER üzerinde) ---
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

# --- NB21 Orijinal: Özdeş çift sütunlar (MASTER üzerinde) ---
def get_duplicate_col_pairs_nb21(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs_nb21(df_master, num_feat)
nb21_drop = set(constant_cols) | dup_drop

print(f"=== NB21 Orijinal Temizlik ===")
print(f"  Constant: {len(constant_cols)}, Duplicate pairs: {len(dup_pairs)} → drop {len(dup_drop)}")
print(f"  NB21 toplam drop: {len(nb21_drop)}")

# --- EDA EK ADIM 1: Hayalet sütunlar (dolu < 30, COMBINED üzerinde tespit) ---
MIN_NON_NULL = 30
ghost_cols = [c for c in feat_cols 
              if c not in nb21_drop 
              and df_combined[c].notnull().sum() < MIN_NON_NULL]
print(f"\n=== EDA Ek Adım 1: Hayalet Sütunlar ===")
print(f"  Dolu < {MIN_NON_NULL} satır: {len(ghost_cols)} sütun")
if ghost_cols:
    print(f"  {ghost_cols[:15]}{'...' if len(ghost_cols) > 15 else ''}")

# --- EDA EK ADIM 2: Multicollinearity (r > 0.90, COMBINED üzerinde tespit) ---
CORR_THRESHOLD = 0.90
keep_after_nb21_ghost = [c for c in feat_cols if c not in nb21_drop and c not in ghost_cols]
numeric_keep = [c for c in keep_after_nb21_ghost 
                if df_combined[c].dtype in [np.float64, np.int64, float, int]]

corr_matrix = df_combined[numeric_keep].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_drop = [col for col in upper_tri.columns if any(upper_tri[col] > CORR_THRESHOLD)]

print(f"\n=== EDA Ek Adım 2: Multicollinearity ===")
print(f"  r > {CORR_THRESHOLD}: {len(high_corr_drop)} sütun drop")

# --- Birleştir: tüm drop sütunları ---
all_drop = nb21_drop | set(ghost_cols) | set(high_corr_drop)
keep_cols = [c for c in feat_cols if c not in all_drop]

print(f"\n=== Toplam ===")
print(f"  Ham feature: {len(feat_cols)}")
print(f"  NB21 drop: {len(nb21_drop)}")
print(f"  EDA hayalet drop: {len(ghost_cols)}")
print(f"  EDA multicoll drop: {len(high_corr_drop)}")
print(f"  Toplam drop: {len(all_drop)}")
print(f"  Kalan feature: {len(keep_cols)}")
print(f"  NB21 orijinal kalıyordu: {len(feat_cols) - len(nb21_drop)}")
print(f"  EDA farkı: {len(ghost_cols) + len(high_corr_drop)} ek sütun temizlendi")

# Her dataset'ten drop
df_master_c  = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser_c  = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr_c    = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah_c     = df_pah[[ID_COL, TARGET] + keep_cols].copy()
df_combined_c = pd.concat([df_master_c, df_kanser_c, df_cftr_c], ignore_index=True)

print(f"\nFinal: COMBINED={df_combined_c.shape}, PAH={df_pah_c.shape}")

=== NB21 Orijinal Temizlik ===
  Constant: 0, Duplicate pairs: 58 → drop 58
  NB21 toplam drop: 58

=== EDA Ek Adım 1: Hayalet Sütunlar ===
  Dolu < 30 satır: 0 sütun

=== EDA Ek Adım 2: Multicollinearity ===
  r > 0.9: 117 sütun drop

=== Toplam ===
  Ham feature: 351
  NB21 drop: 58
  EDA hayalet drop: 0
  EDA multicoll drop: 117
  Toplam drop: 175
  Kalan feature: 176
  NB21 orijinal kalıyordu: 293
  EDA farkı: 117 ek sütun temizlendi

Final: COMBINED=(3430, 178), PAH=(369, 178)


## Faz 2: M3 Preprocessing + Değerlendirme Altyapısı (NB21 ile aynı)

In [4]:
# Cell 4: M3 Preprocessing (NB21 ile birebir aynı — hiçbir şey değişmedi)
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols, target):
    X = train_df[keep_cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    medians = X[num_cols].median()
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians, "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    X = df[keep_cols].copy()
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    for c in prep["num_cols"]:
        X[c] = X[c].fillna(prep["medians"][c])
    for c in prep["cat_cols"]:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return X

# Preprocessor fit (NB21: COMBINED üzerinde)
prep_combined = fit_preprocessor(df_combined_c, keep_cols, TARGET)

# Transform
X_combined_df = transform_X(df_combined_c, keep_cols, prep_combined)
y_combined = df_combined_c[TARGET].values

X_pah_df = transform_X(df_pah_c, keep_cols, prep_combined)
y_pah = df_pah_c[TARGET].values

print(f"X_combined: {X_combined_df.shape}")
print(f"X_pah: {X_pah_df.shape}")
print(f"PAH label: pos={y_pah.sum()}, neg={(y_pah==0).sum()}")
print(f"is_missing flag sayısı: {len(prep_combined['high_miss'])}")

X_combined: (3430, 260)
X_pah: (369, 260)
PAH label: pos=307, neg=62
is_missing flag sayısı: 84


In [5]:
# Cell 5: Değerlendirme Altyapısı (NB21 ile birebir aynı)

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y, prob = np.asarray(y), np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def loo_metrics(y_true, oof_proba, prior_shift=False, pi_train=None):
    if pi_train is None:
        pi_train = y_true.mean()
    prob = adjust_prior_shift(oof_proba, pi_train=pi_train) if prior_shift else oof_proba
    thr = select_threshold_8020_robust(y_true, prob)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1 = _f1_pos(y_true, y_pred)
    auc = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {"mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

print("Değerlendirme altyapısı hazır.")

Değerlendirme altyapısı hazır.


In [6]:
# Cell 6: Model Parametreleri (NB21 ile birebir aynı)

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

def _lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

print("Model parametreleri hazır (NB21 ile aynı).")

Model parametreleri hazır (NB21 ile aynı).


## Faz 3: NB21 Stratejileri — EDA-Temizlikli Veri ile

NB21'in en iyi 3 stratejisini (P0c COMBINED, P2 BalancedBagging, P4 COMBINED+BalBag) aynen çalıştır.
Model parametreleri, değerlendirme protokolü **hiç değişmedi** — sadece girdi feature seti EDA ile daraltıldı.

In [7]:
# Cell 7: 4 Strateji — NB21 ile aynı model, EDA-temizlikli veri
print("=" * 70)
print("NB42 — PAH: NB21 Stratejileri + EDA Preprocessing")
print("=" * 70)

all_results = {}
pi_combined = float(y_combined.mean())

# Pre-encode
X_combined_le = X_combined_df.copy()
X_pah_le = X_pah_df.copy()
cat_cols_le = X_combined_le.select_dtypes(include=["object", "category"]).columns.tolist()
le_maps_enc = {}
for c in cat_cols_le:
    X_combined_le[c] = X_combined_le[c].fillna("MISSING").astype(str)
    X_pah_le[c] = X_pah_le[c].fillna("MISSING").astype(str)
    le = LabelEncoder()
    le.fit(pd.concat([X_combined_le[c], X_pah_le[c]]))
    X_combined_le[c] = le.transform(X_combined_le[c])
    X_pah_le[c] = le.transform(X_pah_le[c])
    le_maps_enc[c] = le

# ================================================================
# P0c_EDA: COMBINED (NB21 P0c + EDA temizlik)
# ================================================================
print("\nP0c_EDA: COMBINED (MASTER+KANSER+CFTR) + EDA preprocessing...")
m_p0c = _lgbm_classifier()
m_p0c.fit(X_combined_le, y_combined)
p_p0c_train = m_p0c.predict_proba(X_combined_le)[:, 1]
p_p0c_pah = m_p0c.predict_proba(X_pah_le)[:, 1]

thr_p0c = select_threshold_8020_robust(y_combined, p_p0c_train)
train_p0c = train_metrics_at(y_combined, p_p0c_train, thr_p0c)
loo_p0c_raw = loo_metrics(y_pah, p_p0c_pah, prior_shift=False)
loo_p0c_prior = loo_metrics(y_pah, p_p0c_pah, prior_shift=True, pi_train=pi_combined)

all_results["P0c_EDA_COMBINED"] = {
    "loo_raw": loo_p0c_raw, "loo_prior": loo_p0c_prior,
    "train": train_p0c, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_p0c_raw['mcc']:.4f} MCC(prior)={loo_p0c_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p0c_prior['boot8020']['mean']:.4f} ± {loo_p0c_prior['boot8020']['std']:.4f}")

# ================================================================
# P2_EDA: BalancedBagging (COMBINED üzerinde — NB21'de MASTER'daydı ama
# karşılaştırma için COMBINED ile de deneyelim)
# ================================================================
print("\nP2_EDA: BalancedBagging (COMBINED, 20x balanced) + EDA preprocessing...")
bb_p2 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_p2.fit(X_combined_le, y_combined)
p_p2_train = bb_p2.predict_proba(X_combined_le)[:, 1]
p_p2_pah = bb_p2.predict_proba(X_pah_le)[:, 1]

thr_p2 = select_threshold_8020_robust(y_combined, p_p2_train)
train_p2 = train_metrics_at(y_combined, p_p2_train, thr_p2)
loo_p2_raw = loo_metrics(y_pah, p_p2_pah, prior_shift=False)
loo_p2_prior = loo_metrics(y_pah, p_p2_pah, prior_shift=True, pi_train=pi_combined)

all_results["P2_EDA_BalBag_COMBINED"] = {
    "loo_raw": loo_p2_raw, "loo_prior": loo_p2_prior,
    "train": train_p2, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_p2_raw['mcc']:.4f} MCC(prior)={loo_p2_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p2_prior['boot8020']['mean']:.4f} ± {loo_p2_prior['boot8020']['std']:.4f}")

# ================================================================
# P4_EDA: COMBINED + BalancedBagging (NB21'in kazanan stratejisi)
# ================================================================
print("\nP4_EDA: COMBINED + BalancedBagging (NB21 kazanan, + EDA)...")
bb_p4 = BalancedBaggingClassifier(
    estimator=_lgbm_classifier(),
    n_estimators=20,
    sampling_strategy="not minority",
    random_state=SEED,
    n_jobs=-1
)
bb_p4.fit(X_combined_le, y_combined)
p_p4_train = bb_p4.predict_proba(X_combined_le)[:, 1]
p_p4_pah = bb_p4.predict_proba(X_pah_le)[:, 1]

thr_p4 = select_threshold_8020_robust(y_combined, p_p4_train)
train_p4 = train_metrics_at(y_combined, p_p4_train, thr_p4)
loo_p4_raw = loo_metrics(y_pah, p_p4_pah, prior_shift=False)
loo_p4_prior = loo_metrics(y_pah, p_p4_pah, prior_shift=True, pi_train=pi_combined)

all_results["P4_EDA_COMBINED_BalBag"] = {
    "loo_raw": loo_p4_raw, "loo_prior": loo_p4_prior,
    "train": train_p4, "pi_train": pi_combined, "n_train": len(y_combined)
}
print(f"  MCC(raw)={loo_p4_raw['mcc']:.4f} MCC(prior)={loo_p4_prior['mcc']:.4f}")
print(f"  Boot-mean(prior)={loo_p4_prior['boot8020']['mean']:.4f} ± {loo_p4_prior['boot8020']['std']:.4f}")

# ================================================================
# P4c_EDA_PriorShift: P4 + Saerens prior-shift post-hoc
# ================================================================
print("\nP4c_EDA_PriorShift: P4 + Saerens post-hoc...")
all_results["P4c_EDA_PriorShift"] = {
    "loo_raw": loo_p4_raw, "loo_prior": loo_p4_prior,
    "train": train_p4, "pi_train": pi_combined, "n_train": len(y_combined),
    "note": "Same model as P4_EDA, prior-shift applied post-hoc"
}
print(f"  MCC={loo_p4_prior['mcc']:.4f} Boot-mean={loo_p4_prior['boot8020']['mean']:.4f}")

print("\n" + "=" * 70)
print("Tüm stratejiler tamamlandı!")
print("=" * 70)

NB42 — PAH: NB21 Stratejileri + EDA Preprocessing

P0c_EDA: COMBINED (MASTER+KANSER+CFTR) + EDA preprocessing...
  MCC(raw)=0.5217 MCC(prior)=0.4904
  Boot-mean(prior)=0.5669 ± 0.0436

P2_EDA: BalancedBagging (COMBINED, 20x balanced) + EDA preprocessing...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC(raw)=0.4526 MCC(prior)=0.4604
  Boot-mean(prior)=0.5402 ± 0.0485

P4_EDA: COMBINED + BalancedBagging (NB21 kazanan, + EDA)...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC(raw)=0.4526 MCC(prior)=0.4604
  Boot-mean(prior)=0.5402 ± 0.0485

P4c_EDA_PriorShift: P4 + Saerens post-hoc...
  MCC=0.4604 Boot-mean=0.5402

Tüm stratejiler tamamlandı!


## Faz 4: NB21 Orijinal vs NB42 EDA Karşılaştırma

In [8]:
# Cell 8: Sonuç Tablosu + NB21 Referans Karşılaştırma

# NB21 orijinal sonuçları (sabit referans — notebook çıktısından)
nb21_ref = {
    "P4_COMBINED_BalBag": {"boot_mean": 0.582, "mcc": 0.529, "boot_lo": 0.45, "boot_hi": 0.70},
    "P0c_COMBINED":       {"boot_mean": 0.543, "mcc": 0.490, "boot_lo": 0.40, "boot_hi": 0.65},
    "P2_BalancedBagging":  {"boot_mean": 0.520, "mcc": 0.470, "boot_lo": 0.38, "boot_hi": 0.63},
}

# Floor F1
prev_8020 = 0.20
floor_f1 = 2 * prev_8020 / (1 + prev_8020)

rows = []
for name, res in all_results.items():
    loo_p = res["loo_prior"]
    boot = loo_p["boot8020"]
    rows.append({
        "Strateji": name,
        "n_train": res["n_train"],
        "MCC(prior)": round(loo_p["mcc"], 4),
        "Boot-mean": round(boot["mean"], 4),
        "Boot-std": round(boot["std"], 4),
        "CI": f"[{boot['lo']:.3f}–{boot['hi']:.3f}]",
        "Precision": round(loo_p["precision"], 4),
        "Recall": round(loo_p["recall"], 4),
        "TN": loo_p["tn"], "FP": loo_p["fp"],
        "FN": loo_p["fn"], "TP": loo_p["tp"],
        "Floor": floor_f1
    })

df_nb42 = pd.DataFrame(rows).sort_values("Boot-mean", ascending=False).reset_index(drop=True)

print("=" * 80)
print("NB42 EDA SONUÇLARI (prior-shift, Boot %80/20)")
print("=" * 80)
print(df_nb42[["Strateji", "MCC(prior)", "Boot-mean", "CI", "Precision", "Recall"]].to_string(index=False))

print(f"\n{'='*80}")
print("NB21 ORİJİNAL vs NB42 EDA KARŞILAŞTIRMA")
print(f"{'='*80}")
print(f"{'Strateji':<30} {'NB21 Boot':>10} {'NB42 Boot':>10} {'Δ':>8} {'Durum':>10}")
print("-" * 70)

comparisons = [
    ("P4_COMBINED_BalBag", "P4_EDA_COMBINED_BalBag"),
    ("P0c_COMBINED", "P0c_EDA_COMBINED"),
]
for nb21_key, nb42_key in comparisons:
    nb21_val = nb21_ref[nb21_key]["boot_mean"]
    nb42_res = all_results.get(nb42_key)
    if nb42_res:
        nb42_val = nb42_res["loo_prior"]["boot8020"]["mean"]
        delta = nb42_val - nb21_val
        status = "↑ İYİ" if delta > 0.01 else ("≈ AYNI" if abs(delta) <= 0.01 else "↓ KÖTÜ")
        print(f"{nb21_key:<30} {nb21_val:>10.4f} {nb42_val:>10.4f} {delta:>+8.4f} {status:>10}")

print(f"\nFloor F1 (%80/20): {floor_f1:.4f}")
print(f"NB21 en iyi: P4_COMBINED_BalBag Boot={nb21_ref['P4_COMBINED_BalBag']['boot_mean']:.4f}")

# CSV kaydet
results_file = os.path.join(RESULTS_DIR_NB, "nb42_eda_combined_results.csv")
df_nb42.to_csv(results_file, index=False)
print(f"\nSonuçlar: {results_file}")

NB42 EDA SONUÇLARI (prior-shift, Boot %80/20)
              Strateji  MCC(prior)  Boot-mean            CI  Precision  Recall
      P0c_EDA_COMBINED      0.4904     0.5669 [0.476–0.638]     0.9384  0.8436
P2_EDA_BalBag_COMBINED      0.4604     0.5402 [0.465–0.618]     0.9341  0.8306
P4_EDA_COMBINED_BalBag      0.4604     0.5402 [0.465–0.618]     0.9341  0.8306
    P4c_EDA_PriorShift      0.4604     0.5402 [0.465–0.618]     0.9341  0.8306

NB21 ORİJİNAL vs NB42 EDA KARŞILAŞTIRMA
Strateji                        NB21 Boot  NB42 Boot        Δ      Durum
----------------------------------------------------------------------
P4_COMBINED_BalBag                 0.5820     0.5402  -0.0418     ↓ KÖTÜ
P0c_COMBINED                       0.5430     0.5669  +0.0239      ↑ İYİ

Floor F1 (%80/20): 0.3333
NB21 en iyi: P4_COMBINED_BalBag Boot=0.5820

Sonuçlar: /Users/tefe/teknofest_model/teknofest_model/results/v25_pah_eda_pipeline/nb42_eda_combined_results.csv


## Faz 5: Özet ve Sonuç

In [9]:
# Cell 9: Özet
print("=" * 70)
print("NB42 ÖZET")
print("=" * 70)

best_nb42 = df_nb42.iloc[0]
print(f"\nEn iyi NB42 stratejisi: {best_nb42['Strateji']}")
print(f"  Boot-mean: {best_nb42['Boot-mean']}")
print(f"  MCC: {best_nb42['MCC(prior)']}")
print(f"  CI: {best_nb42['CI']}")
print(f"  Precision: {best_nb42['Precision']}, Recall: {best_nb42['Recall']}")

print(f"\n--- EDA Preprocessing Etkisi ---")
print(f"  NB21 feature sayısı: {len(feat_cols) - len(nb21_drop)} (sabit+dup drop)")
print(f"  NB42 feature sayısı: {len(keep_cols)} (+ hayalet + multicollinearity drop)")
print(f"  Ek temizlenen: {(len(feat_cols) - len(nb21_drop)) - len(keep_cols)} sütun")

print(f"\n--- PAH Chatterji Tavanı ---")
print(f"  Floor F1 (train): {2 * y_pah.mean() / (1 + y_pah.mean()):.4f}")
print(f"  Floor F1 (%80/20): {floor_f1:.4f}")
print(f"  Danışman best: 0.925 ≈ floor 0.905")
print(f"  NB21 en iyi: 0.582")
print(f"  PAH KESİNLEŞTİ — yeni model arama kapalı (CLAUDE.md)")

NB42 ÖZET

En iyi NB42 stratejisi: P0c_EDA_COMBINED
  Boot-mean: 0.5669
  MCC: 0.4904
  CI: [0.476–0.638]
  Precision: 0.9384, Recall: 0.8436

--- EDA Preprocessing Etkisi ---
  NB21 feature sayısı: 293 (sabit+dup drop)
  NB42 feature sayısı: 176 (+ hayalet + multicollinearity drop)
  Ek temizlenen: 117 sütun

--- PAH Chatterji Tavanı ---
  Floor F1 (train): 0.9083
  Floor F1 (%80/20): 0.3333
  Danışman best: 0.925 ≈ floor 0.905
  NB21 en iyi: 0.582
  PAH KESİNLEŞTİ — yeni model arama kapalı (CLAUDE.md)
